# 08 — Validación Clínica — Bland-Altman / ICC vs AVA 3.2

Comparación de métricas automáticas contra el gold standard del xlsx.
Análisis de concordancia: Bland-Altman, ICC, curvas ROC.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy import stats

DATA_PATH   = Path(os.environ.get("MICROCIRCULATION_DATA", "/content/drive/MyDrive/microcirculation"))
XLSX_PATH   = DATA_PATH / "Grilla_trauma_SL_.xlsx"


## 8.1 Cargar gold standard (xlsx) y métricas automáticas

In [ ]:
# ── Gold standard ─────────────────────────────────────────────────────────────
raw = pd.read_excel(XLSX_PATH, header=1)
raw.columns = (
    ['paciente','video','flow_0','flow_1','flow_2','flow_3','TOT',
     'small_vessel_density','TVD']
    + [f'V{i}' for i in range(1,21)] + ['observaciones']
)
raw['paciente'] = raw['paciente'].fillna(method='ffill')
VEL_COLS = [f'V{i}' for i in range(1, 21)]

gs = raw[raw['TVD'].notna()].copy()
gs['PPV']      = gs['flow_3'] / gs['TOT']
gs['vel_mean'] = gs[VEL_COLS].mean(axis=1)
gs['HI']       = gs[VEL_COLS].std(axis=1)
gs['n_vessels']= gs[VEL_COLS].notna().sum(axis=1)

BAD = ['foco','inestable','back','compresión','compresion']
gs['quality_ok'] = ~gs['observaciones'].str.lower().str.contains(
    '|'.join(BAD), na=False)

# Clave de unión: nombre del video (D1-V1, etc.)
gs['video_key'] = gs['video'].str.replace(r'\(.*\)', '', regex=True).str.strip()

# ── Métricas automáticas ──────────────────────────────────────────────────────
auto = pd.read_csv(DATA_PATH / "auto_metrics.csv")
# TODO: ajustar el nombre de la columna 'video' para que coincida con 'video_key'
# auto['video_key'] = auto['video'].apply(lambda x: ...)

print(f"Gold standard: {len(gs)} videos")
print(f"Automático:    {len(auto)} videos")


## 8.2 Bland-Altman — función genérica

In [ ]:
def bland_altman_analysis(method_a: pd.Series, method_b: pd.Series,
                           label_a: str = "AVA 3.2 (gold)",
                           label_b: str = "Pipeline auto",
                           unit: str = "") -> dict:
    """
    Calcula y grafica el análisis de Bland-Altman.
    Retorna dict con bias, SD, LOA inferior y superior.
    """
    diff    = method_a - method_b
    mean_ab = (method_a + method_b) / 2
    bias    = diff.mean()
    sd      = diff.std()
    loa_lo  = bias - 1.96 * sd
    loa_hi  = bias + 1.96 * sd

    # IC 95% de los LOA (Bland & Altman 1999)
    n       = len(diff)
    se_loa  = np.sqrt(3 * sd**2 / n)
    ci_lo   = (loa_lo - 1.96*se_loa, loa_lo + 1.96*se_loa)
    ci_hi   = (loa_hi - 1.96*se_loa, loa_hi + 1.96*se_loa)

    # Gráfico
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(mean_ab, diff, alpha=0.7, s=40, color='steelblue', zorder=3)
    ax.axhline(bias,   color='steelblue', lw=2,   label=f"Bias = {bias:.3f} {unit}")
    ax.axhline(loa_hi, color='tomato',    lw=1.5, linestyle='--',
               label=f"LOA+ = {loa_hi:.3f} {unit}")
    ax.axhline(loa_lo, color='tomato',    lw=1.5, linestyle='--',
               label=f"LOA− = {loa_lo:.3f} {unit}")
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')

    # IC de LOA
    ax.fill_between(ax.get_xlim(), ci_hi[0], ci_hi[1], alpha=0.08, color='tomato')
    ax.fill_between(ax.get_xlim(), ci_lo[0], ci_lo[1], alpha=0.08, color='tomato')

    ax.set_xlabel(f"Promedio ({label_a}, {label_b}) [{unit}]")
    ax.set_ylabel(f"Diferencia ({label_a} − {label_b}) [{unit}]")
    ax.set_title(f"Bland-Altman — {label_b}")
    ax.legend(fontsize=9); ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(DATA_PATH / f"fig_08_ba_{label_b.replace(' ','_')}.png", dpi=150)
    plt.show()

    print(f"\n{'='*45}")
    print(f"Bland-Altman: {label_b}  [{unit}]")
    print(f"  Bias:   {bias:+.4f}")
    print(f"  SD:     {sd:.4f}")
    print(f"  LOA:    [{loa_lo:+.4f}, {loa_hi:+.4f}]")
    print(f"  IC LOA-:[{ci_lo[0]:+.3f}, {ci_lo[1]:+.3f}]")
    print(f"  IC LOA+:[{ci_hi[0]:+.3f}, {ci_hi[1]:+.3f}]")

    return {"bias":bias,"sd":sd,"loa_lo":loa_lo,"loa_hi":loa_hi,
            "ci_lo":ci_lo,"ci_hi":ci_hi,"n":n}


## 8.3 ICC — Intraclass Correlation Coefficient

In [ ]:
def compute_icc(method_a: pd.Series, method_b: pd.Series) -> dict:
    """
    ICC(2,1) — two-way mixed, absolute agreement.
    Fórmula: Shrout & Fleiss (1979).
    """
    data = pd.DataFrame({"a": method_a, "b": method_b}).dropna()
    n    = len(data)
    if n < 3:
        return {"icc": np.nan, "ci_lo": np.nan, "ci_hi": np.nan}

    grand_mean = data.values.mean()
    row_means  = data.mean(axis=1)
    col_means  = data.mean(axis=0)

    SS_r = 2 * ((row_means - grand_mean)**2).sum()
    SS_c = n  * ((col_means - grand_mean)**2).sum()
    SS_e = ((data.subtract(row_means, axis=0)
                 .subtract(col_means, axis=1)
                 + grand_mean)**2).values.sum()

    MS_r = SS_r / (n - 1)
    MS_c = SS_c
    MS_e = SS_e / (n - 1)

    icc  = (MS_r - MS_e) / (MS_r + MS_c + (2 - 1) * MS_e - MS_c / n)

    # CI aproximado (Fisher Z)
    z    = 0.5 * np.log((1 + icc) / (1 - icc + 1e-9))
    se_z = 1 / np.sqrt(n - 3) if n > 3 else np.inf
    ci_lo = np.tanh(z - 1.96 * se_z)
    ci_hi = np.tanh(z + 1.96 * se_z)

    return {"icc": round(icc, 4), "ci_lo": round(ci_lo, 4), "ci_hi": round(ci_hi, 4)}


## 8.4 Ejecutar validación para cada métrica

In [ ]:
# ── Merge gold standard y automático ─────────────────────────────────────────
# TODO: asegurar que 'video_key' sea la misma columna en ambos dataframes
# merged = gs.merge(auto, left_on='video_key', right_on='video', how='inner')
# Por ahora, ejemplo sintético para demostrar el flujo
# Reemplazar con el merge real una vez que estén los datos automáticos

# merged = merged[merged['quality_ok']]  # solo videos de buena calidad

results = {}
pairs = [
    ("TVD",      "TVD_auto",      "mm/mm²"),
    ("PPV",      "PPV_auto",      "ratio"),
    ("vel_mean", "vel_mean_auto", "µm/s"),
    ("HI",       "HI_auto",       "µm/s"),
]

# for col_gs, col_auto, unit in pairs:
#     a = merged[col_gs]
#     b = merged[col_auto]
#     ba  = bland_altman_analysis(a, b, unit=unit)
#     icc = compute_icc(a, b)
#     results[col_gs] = {**ba, **icc}
#     print(f"ICC {col_gs}: {icc['icc']:.3f} [{icc['ci_lo']:.3f}, {icc['ci_hi']:.3f}]")

print("Celda lista. Descomentar cuando el merge de datos esté disponible.")
print("Pares a analizar:", [p[0] for p in pairs])


## 8.5 Curvas ROC — clasificación perfusión normal/alterada

In [ ]:
from sklearn.metrics import roc_curve, auc

# Umbral clínico de PPV (consenso ESICM: < 0.70 = alterado)
PPV_THRESHOLD = 0.70

# gs['perfusion_impaired'] = (gs['PPV'] < PPV_THRESHOLD).astype(int)
# fpr, tpr, _ = roc_curve(gs['perfusion_impaired'], merged['PPV_auto'])
# roc_auc = auc(fpr, tpr)

# plt.figure(figsize=(6,6))
# plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}", color='steelblue', lw=2)
# plt.plot([0,1],[0,1],'--', color='gray')
# plt.xlabel("1 - Especificidad"); plt.ylabel("Sensibilidad")
# plt.title("ROC — PPV_auto para clasificar perfusión alterada")
# plt.legend(); plt.tight_layout()
# plt.savefig(DATA_PATH / "fig_08_roc_ppv.png", dpi=150)
# plt.show()

print("ROC lista. Descomentar con datos reales.")
print(f"Umbral PPV para perfusión alterada: < {PPV_THRESHOLD}")


## 8.6 Comparación de detección de vasos — n_vessels

In [ ]:
# Punto clave de Dubin et al. (2020): MicroTools detectaba ~36 vasos vs ~200 de AVA
# Tu pipeline debe reportar n_vessels comparable al gold standard

# if 'n_vessels' in merged.columns and 'n_vessels_auto' in merged.columns:
#     print("Vasos detectados — comparación:")
#     print(f"  AVA 3.2 (gold):      {merged['n_vessels'].median():.0f} vasos/video (mediana)")
#     print(f"  Pipeline automático: {merged['n_vessels_auto'].median():.0f} vasos/video (mediana)")
#     ratio = merged['n_vessels_auto'].median() / merged['n_vessels'].median()
#     print(f"  Ratio auto/gold:     {ratio:.2f}  (objetivo: > 0.5)")

print("Pendiente merge. Criterio de éxito: ratio auto/gold > 0.5 (superar MicroTools)")
print("MicroTools: 36/200 = 0.18")
